<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#isnull" data-toc-modified-id="isnull-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>isnull</a></span><ul class="toc-item"><li><span><a href="#na" data-toc-modified-id="na-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>na</a></span></li><li><span><a href="#outer" data-toc-modified-id="outer-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>outer</a></span></li></ul></li></ul></div>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType
import os, warnings

warnings.filterwarnings(action="ignore")

In [2]:
spark = (SparkSession.builder
         .appName("05-API.DataFrames-donnees-manquantes")
         .getOrCreate())

print("Master :", spark.sparkContext.master)
print("Application :", spark.sparkContext.applicationId)
print("Python :", os.sys.version.split()[0])
print("Spark :", spark.version)

26/09/24 10:13:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 10:13:04 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/24 10:13:05 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to spark-events/eventlog_v2_app-20260924101304-0023/events_1_app-20260924101304-0023.zstd. This is Unsupported


Master : spark://spark-master:7077
Application : app-20260924101304-0023
Python : 3.10.12
Spark : 4.0.4


In [3]:
spark

In [4]:
meteoDataFrame  = spark.read.format('csv')\
    .option('sep',';')\
    .option('header','true')\
    .option('nullValue','mq')\
    .option('inferSchema', 'true')\
    .load('../data/meteo/')\
    .cache()

schema = StructType([
        StructField('Id'           , StringType() , True),
        StructField('ville'        , StringType() , True),
        StructField('latitude'     , FloatType() , True),
        StructField('longitude'    , FloatType() , True),
        StructField('altitude'     , IntegerType() , True)])

villes  = spark.read.format('csv')   \
      .option('sep',';')                \
      .option('mergeSchema', 'true')    \
      .option('header','true')          \
      .schema(schema)                   \
      .load('../data/postesSynop.csv')  \
      .cache()

@udf("string")
def formatVille(ville):
    if ville in ['CLERMONT-FD','MONT-DE-MARSAN',
                                   'ST-PIERRE','ST-BARTHELEMY METEO'] :
        return ville.title()
    else :
        if ville.find('-') != -1 :
            return ville[0:ville.find('-')].title()
        else:
            return ville.title()

villesT  = villes.select(
                col('Id').alias('id'),
                formatVille('ville').alias('ville'),
               'latitude',
               'longitude',
               'altitude')


meteo = meteoDataFrame.select(
                 col('numer_sta'),
                 col('date')[0:4].cast('int') ,
                 col('date')[5:2].cast('int'),
                 col('date')[7:2].cast('int'),
                 col('date')[5:4],
                 round(col('t') - 273.15,2),
                 col('u') / 100 ,
                 col('vv') / 1000 ,
                 col('pres') / 1000,
                 coalesce( col('rr3'),
                           col('rr24')/8,
                           col('rr12')/4,
                           col('rr6')/2,
                           col('rr1')*3  ) )\
             .toDF('id','annee','mois','jour','mois_jour','temperature',
                   'humidite','visibilite','pression','precipitations')\
             .cache()

meteo.select('annee','mois','jour','temperature','humidite',
             'visibilite','pression').toPandas().head(3)

26/09/24 10:13:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

,annee,mois,jour,temperature,humidite,visibilite,pression
0,2024,5,1,13.1,0.94,14.20,100.13
1,2024,5,1,14.5,0.85,16.03,100.56
2,2024,5,1,8.8,0.88,NaN,100.74


In [5]:
villesT.toPandas().head(3)

,id,ville,latitude,longitude,altitude
0,07005,Abbeville,50.136002,1.834000,69
1,07015,Lille,50.570000,3.097500,47
2,07020,Pte De La Hague,49.725166,-1.939833,6


# isnull

In [6]:
meteo.where('id < 8000').count()
description = meteo.where('id < 8000')\
     .select('annee','temperature', 'humidite',
             'visibilite', 'pression','precipitations')\
     .describe()\
     .select([c if c == 'summary'
                else round(c,2).alias(c)
                for c in
                    ['summary','annee','temperature', 'humidite',
                   'visibilite', 'pression','precipitations']])\
     .toPandas()

In [7]:
meteo.where('id < 8000').where(meteo['temperature'].isNull()).count()

9261

In [8]:
meteo.where('id < 8000').where(meteo['humidite'].isNull()).count()

12429

In [9]:
meteo.where('id < 8000').where(meteo['visibilite'].isNull()).count()

30627

In [10]:
meteo.where('id < 8000').where(meteo['pression'].isNull()).count()

12074

In [11]:
meteo.where('id < 8000').where(meteo['precipitations'].isNull()).count()

10024

In [12]:
meteo.where('id < 8000').toPandas().isna().sum()

id                    0
annee                 0
mois                  0
jour                  0
mois_jour             0
temperature        9261
humidite          12429
visibilite        30627
pression          12074
precipitations    10024
dtype: int64

In [13]:
meteoDataFrame.toPandas().isna().sum()

numer_sta         0
date              0
pmer          36033
tend          14729
cod_tend      14724
dd             3018
ff             3018
t             12422
td            18799
u             16956
vv           134792
ww           141943
w1           469086
w2           494761
n            320197
nbas         178535
hbas         272885
cl           476767
cm           489908
ch           495413
pres          14363
niv_bar      510645
geop         510645
tend24        28938
tn12         429510
tn24         422171
tx12         431473
tx24         421588
tminsol      214775
sw           525327
tw           525327
raf10         95787
rafper        49450
per           46487
etat_sol     297557
ht_neige     230376
ssfrai       512311
perssfrai    512311
rr1           19431
rr3           23362
rr6           28634
rr12          33071
rr24          38968
phenspe1     525327
phenspe2     525327
phenspe3     525327
phenspe4     525327
nnuage1      268053
ctype1       475755
hnuage1      273747


In [14]:
meteo.where('id < 8000')\
     .where(meteo['temperature'].isNotNull())\
     .where(meteo['humidite'].isNotNull() )\
     .where(meteo['visibilite'].isNotNull() )\
     .where(meteo['pression'].isNotNull() )\
     .count()

328227

## na

In [15]:
meteo.where('id < 8000')\
     .na.fill(0 ,["precipitations"])\
     .na.drop()\
     .count()

328227

In [16]:
meteo = meteoDataFrame.select(
                 col('numer_sta'),
                 col('date')[0:4].cast('int') ,
                 col('date')[5:2].cast('int'),
                 col('date')[7:2].cast('int'),
                 col('date')[5:4],
                 round(col('t') - 273.15,2),
                 col('u') / 100 ,
                 col('vv') / 1000 ,
                 col('pres') / 1000,
                 col('rr1')*3,
                 col('rr3'),
                 col('rr6')/2,
                 col('rr12')/4,
                 col('rr24')/8)\
                   .toDF('id','annee','mois','jour','mois_jour','temperature',
                   'humidite','visibilite','pression',
                   'precipitations1','precipitations3','precipitations6',
                   'precipitations12','precipitations24')\
             .cache()

In [17]:
meteo.where('id < 8000')\
       .select('precipitations1','precipitations3','precipitations6',
             'precipitations12','precipitations24','temperature')\
       .toDF('prec1','prec3','prec6','prec12','prec24','temp')\
       .describe()\
       .select([c if c == 'summary'
                else round(c,2).alias(c)
                for c in['prec1','prec3','prec6',
                          'prec12','prec24','temp']])\
      .toPandas()

,prec1,prec3,prec6,prec12,prec24,temp
0,353135.00,353143.00,350331.00,347999.00,343481.00,354933.00
1,0.25,0.25,0.26,0.26,0.26,13.52
2,1.66,1.24,1.03,0.82,0.65,7.26
3,-0.30,-0.10,-0.05,-0.03,-0.01,-12.10
4,120.00,63.00,41.95,29.25,16.59,42.30


In [18]:
meteo.select( coalesce('precipitations3','precipitations24',
                       'precipitations12','precipitations6',
                       'precipitations1').alias('precipitations')).toPandas().head(20)

,precipitations
0,0.2
1,-0.1
2,0.0
3,0.0
4,0.0
5,1.2
6,-0.1
7,-0.1
8,0.2
9,0.0


In [19]:
meteo.where('id < 8000')\
     .select( coalesce('precipitations24','precipitations12',
                       'precipitations6','precipitations3',
                       'precipitations1').alias('precipitationsH')
            ).describe().toPandas()

,summary,precipitationsH
0,count,354170
1,mean,0.26111588361518934
2,stddev,0.6806271059320227
3,min,-0.30000000000000004
4,max,42.3


In [20]:
meteo.toPandas().isna().sum()

id                       0
annee                    0
mois                     0
jour                     0
mois_jour                0
temperature          12422
humidite             16956
visibilite          134792
pression             14363
precipitations1      19431
precipitations3      23362
precipitations6      28634
precipitations12     33071
precipitations24     38968
dtype: int64

In [21]:
meteo.where('id < 8000').count()

364194